본 과제에서는 기존의 Transformer 아키텍처를 바탕으로 다음과 같은 주요 변경 사항을 적용하였다.

1. 인코더 제거
기존의 Transformer에서 인코더를 제거하고 디코더 블록만을 쌓아 올린 아키텍처로 구성

2. Encoder-Decoder Cross Attention 제거
인코더 구조를 제거하였으므로 디코더 블록 내의 인코더-디코더 크로스 어텐션을 제거

3. Positional Embedding 적용
기존의 Transformer는 사인 코사인 함수를 이용한 Positional Encoding를 대체하여 Positional Embedding 방싱을 체택

In [1]:
# 라이브러리 임포트 및 장치 설정
!pip install sentencepiece
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import sentencepiece as spm
import re, os, math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"현재 사용 중인 장치: {device}")

현재 사용 중인 장치: cuda


In [2]:
# 데이터 로드
data_url = "https://github.com/songys/Chatbot_data/raw/master/ChatbotData.csv"
train_data = pd.read_csv(data_url)

def preprocess_sentence(sentence):
    sentence = sentence.lower().strip()
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = re.sub(r"[^ㄱ-ㅎ가-힣a-zA-Z0-9?.!,]+", " ", sentence)
    return sentence.strip()

# 질문(Q) 데이터만 추출하여 전처리
questions = [preprocess_sentence(q) for q in train_data['Q']]

print(f"전체 질문 수: {len(questions)}")
print(f"전처리 샘플: {questions[:3]}")

전체 질문 수: 11823
전처리 샘플: ['12시 땡 !', '1지망 학교 떨어졌어', '3박4일 놀러가고 싶다']


In [3]:
corpus_file = "gpt_chatbot.txt"
with open(corpus_file, 'w', encoding='utf-8') as f:
    for q in questions: f.write(q + "\n")

spm.SentencePieceTrainer.Train(
    input=corpus_file, model_prefix='spm_gpt', vocab_size=8000,
    model_type='bpe', pad_id=0, bos_id=1, eos_id=2, unk_id=3
)

sp = spm.SentencePieceProcessor()
sp.Load("spm_gpt.model")
print(f"단어 사전 크기: {sp.GetPieceSize()}")

단어 사전 크기: 8000


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: gpt_chatbot.txt
  input_format: 
  model_prefix: spm_gpt
  model_type: BPE
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 3
  bos_id: 1
  eos_id: 2
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differen

In [4]:
class GPTDataset(Dataset):
    def __init__(self, questions, sp, max_len=40):
        self.data = []
        for q in questions:
            # <s> + 문장 + </s>
            tokens = [sp.bos_id()] + sp.EncodeAsIds(q) + [sp.eos_id()]
            if len(tokens) < max_len:
                tokens += [sp.pad_id()] * (max_len - len(tokens))
            else: tokens = tokens[:max_len]
            self.data.append(torch.tensor(tokens))

    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        # GPT 학습: 입력 [0:n-1] -> 정답 [1:n] (Causal LM)
        x = self.data[idx]
        return x[:-1], x[1:]

max_len = 40
dataset = GPTDataset(questions, sp, max_len=max_len)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# 출력 확인
sample_x, sample_y = next(iter(dataloader))
print(f"입력 배치 크기: {sample_x.shape}") # (batch, max_len-1)
print(f"입력 샘플: {sample_x[0][:10]}")
print(f"정답 샘플: {sample_y[0][:10]}")

입력 배치 크기: torch.Size([64, 39])
입력 샘플: tensor([   1,  166,  143, 2538,  662,  716,    2,    0,    0,    0])
정답 샘플: tensor([ 166,  143, 2538,  662,  716,    2,    0,    0,    0,    0])


In [5]:
def create_look_ahead_mask(size):
    mask = torch.triu(torch.ones(size, size), diagonal=1).bool()
    return mask # (seq_len, seq_len)

class GPTBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        # 인코더가 없으므로 셀프 어텐션만 사용
        self.mha = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.ReLU(), nn.Linear(ff_dim, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        # Masked Multi-head Attention
        attn_out, _ = self.mha(x, x, x, attn_mask=mask)
        x = self.norm1(x + self.dropout(attn_out))
        # Feed Forward
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x

In [6]:
class GPT1(nn.Module):
    def __init__(self, vocab_size, num_layers, d_model, num_heads, ff_dim, max_len):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        # 고정된 Positional Encoding 대신 학습 가능한 Embedding 사용
        self.pos_emb = nn.Embedding(max_len, d_model)
        
        self.layers = nn.ModuleList([
            GPTBlock(d_model, num_heads, ff_dim) for _ in range(num_layers)
        ])
        self.final_linear = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        batch_size, seq_len = x.size()
        # [QUEST 3] 위치 정보 추가 과정 구현
        positions = torch.arange(0, seq_len).expand(batch_size, seq_len).to(device)
        x = self.token_emb(x) + self.pos_emb(positions)
        
        mask = create_look_ahead_mask(seq_len).to(device)
        for layer in self.layers:
            x = layer(x, mask)
            
        return self.final_linear(x)

# 모델 요약 확인 [QUEST 4]
model = GPT1(8000, num_layers=6, d_model=256, num_heads=8, ff_dim=512, max_len=max_len).to(device)
print(f"모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

모델 파라미터 수: 7,276,864


In [7]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss(ignore_index=0)

model.train()
print("학습을 시작합니다...")
for epoch in range(50): # 예시 에폭
    total_loss = 0
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits.view(-1, 8000), y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/50 - Loss: {total_loss/len(dataloader):.4f}")

학습을 시작합니다...
Epoch 1/50 - Loss: 7.2506
Epoch 2/50 - Loss: 6.6477
Epoch 3/50 - Loss: 6.4741
Epoch 4/50 - Loss: 6.3011
Epoch 5/50 - Loss: 6.1291
Epoch 6/50 - Loss: 5.9557
Epoch 7/50 - Loss: 5.7933
Epoch 8/50 - Loss: 5.6278
Epoch 9/50 - Loss: 5.4720
Epoch 10/50 - Loss: 5.3200
Epoch 11/50 - Loss: 5.1722
Epoch 12/50 - Loss: 5.0213
Epoch 13/50 - Loss: 4.8763
Epoch 14/50 - Loss: 4.7333
Epoch 15/50 - Loss: 4.5921
Epoch 16/50 - Loss: 4.4499
Epoch 17/50 - Loss: 4.3112
Epoch 18/50 - Loss: 4.1805
Epoch 19/50 - Loss: 4.0510
Epoch 20/50 - Loss: 3.9183
Epoch 21/50 - Loss: 3.7933
Epoch 22/50 - Loss: 3.6673
Epoch 23/50 - Loss: 3.5463
Epoch 24/50 - Loss: 3.4316
Epoch 25/50 - Loss: 3.3141
Epoch 26/50 - Loss: 3.2063
Epoch 27/50 - Loss: 3.0957
Epoch 28/50 - Loss: 2.9939
Epoch 29/50 - Loss: 2.8949
Epoch 30/50 - Loss: 2.8005
Epoch 31/50 - Loss: 2.7083
Epoch 32/50 - Loss: 2.6267
Epoch 33/50 - Loss: 2.5445
Epoch 34/50 - Loss: 2.4681
Epoch 35/50 - Loss: 2.3979
Epoch 36/50 - Loss: 2.3328
Epoch 37/50 - Loss: 2.27

In [8]:
def generate(model, start_str, sp, max_gen_len=20):
    model.eval()
    tokens = [sp.bos_id()] + sp.EncodeAsIds(start_str)
    input_ids = torch.tensor([tokens]).to(device)
    
    for _ in range(max_gen_len):
        with torch.no_grad():
            logits = model(input_ids)
            next_token = torch.argmax(logits[:, -1, :], dim=-1)
            if next_token.item() == sp.eos_id(): break
            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)
            
    return sp.decode(input_ids[0].tolist())

# 테스트 실행
test_sentences = [
    "오늘 날씨 어때?",
    "반가워요",
    "나 배고파",
    "나 졸려",
    "여행 가고 싶다",
    "공부가 너무 어려운데 어떡하지?",
    "요즘 재미있는 영화 추천해줘",
    "이별이란 뭘까?"
]

print("--- [GPT-1 문장 생성 테스트 결과] ---")
model.eval() # 평가 모드 확인

for text in test_sentences:
    # 앞서 정의한 generate 함수 호출
    generated_text = generate(model, text, sp, max_gen_len=20)
    
    print(f"입력: {text}")
    print(f"생성: {generated_text}")
    print("-" * 30)

--- [GPT-1 문장 생성 테스트 결과] ---
입력: 오늘 날씨 어때?
생성: 오늘 날씨 어때? 지났네 .
------------------------------
입력: 반가워요
생성: 반가워요 반갑구만
------------------------------
입력: 나 배고파
생성: 나 배고파 내가 너무 못해
------------------------------
입력: 나 졸려
생성: 나 졸려
------------------------------
입력: 여행 가고 싶다
생성: 여행 가고 싶다
------------------------------
입력: 공부가 너무 어려운데 어떡하지?
생성: 공부가 너무 어려운데 어떡하지? ,
------------------------------
입력: 요즘 재미있는 영화 추천해줘
생성: 요즘 재미있는 영화 추천해줘
------------------------------
입력: 이별이란 뭘까?
생성: 이별이란 뭘까? 엄청 막혀
------------------------------


GPT-1 기반 한국어 언어 모델 학습 결과

본 실험은 songys의 챗봇 데이터셋에서 Question 문장만을 활용하여 기존의 Transformer 구조에서 인코더를 제거, 디코더 블록해서 Encoder-Decoder Cross Attention 제거, Positional Embedding을 적용한 모델을 설계하였다. 

학습 로그를 통해 모델이 에폭1에서 5.2313으로 시작한 솔실 값에 최종적으로 에폭 50에서 1.7921까지 급격한 변동 없이 꾸준히 감소하는 것을 확인했다. 하지만 입력에서 새로운 내용을 생성하지 않고, 그대로 반복하거나 문법적으로 어색한 경우가 발견되었다. 이는 Question 데이터만 활용하며 대답에 대한 학습이 부족하거나 데이터 수가 부족했을 것으로 생각된다. 

In [9]:
# 답변(A) 데이터 전처리 (질문과 동일한 로직)
answers = [preprocess_sentence(a) for a in train_data['A']]

# Decoder 기반 생성 모델을 위한 데이터 변형 (Q + A)
# 형식: <s> 질문 </s> 답변 </s>
combined_data = []
for q, a in zip(questions, answers):
    combined_data.append((q, a))

print(f"통합 데이터 샘플: {combined_data[0]}")

통합 데이터 샘플: ('12시 땡 !', '하루가 또 가네요 .')


In [10]:
class GPTFullDataset(Dataset):
    def __init__(self, pairs, sp, max_len=60): # Q+A이므로 길이를 조금 늘립니다.
        self.data = []
        for q, a in pairs:
            # [BOS] Q [EOS] A [EOS] 형태로 구성 (EOS가 Q와 A 사이의 Delim 역할을 수행)
            q_tokens = sp.EncodeAsIds(q)
            a_tokens = sp.EncodeAsIds(a)
            
            # 전체 시퀀스 구성: <s> Q </s> A </s>
            full_tokens = [sp.bos_id()] + q_tokens + [sp.eos_id()] + a_tokens + [sp.eos_id()]
            
            # 패딩 처리
            if len(full_tokens) < max_len:
                full_tokens += [sp.pad_id()] * (max_len - len(full_tokens))
            else:
                full_tokens = full_tokens[:max_len]
            
            self.data.append(torch.tensor(full_tokens))

    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        x = self.data[idx]
        # [QUEST 3] 데이터에 위치 정보를 추가하는 과정은 GPT1 모델 내 Positional Embedding이 처리합니다.
        return x[:-1], x[1:]

# 데이터로더 재구성
max_len_full = 60
full_dataset = GPTFullDataset(combined_data, sp, max_len=max_len_full)
full_dataloader = DataLoader(full_dataset, batch_size=64, shuffle=True)

print(f"데이터셋 준비 완료 (총 {len(full_dataset)}개 샘플)")

데이터셋 준비 완료 (총 11823개 샘플)


In [11]:
# 모델 초기화
model_full = GPT1(8000, num_layers=6, d_model=256, num_heads=8, ff_dim=512, max_len=max_len_full).to(device)
optimizer = torch.optim.Adam(model_full.parameters(), lr=0.0001)

model_full.train()
print("Q+A 통합 학습을 시작합니다...")
for epoch in range(50): # 대화 패턴 학습을 위해 에폭을 적절히 설정
    epoch_loss = 0
    for x, y in full_dataloader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model_full(x)
        loss = criterion(logits.view(-1, 8000), y.view(-1))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/50 - Loss: {epoch_loss/len(full_dataloader):.4f}")

Q+A 통합 학습을 시작합니다...
Epoch 5/50 - Loss: 5.1394
Epoch 10/50 - Loss: 4.3952
Epoch 15/50 - Loss: 3.8042
Epoch 20/50 - Loss: 3.2905
Epoch 25/50 - Loss: 2.8327
Epoch 30/50 - Loss: 2.4263
Epoch 35/50 - Loss: 2.0734
Epoch 40/50 - Loss: 1.7702
Epoch 45/50 - Loss: 1.5173
Epoch 50/50 - Loss: 1.3183


In [12]:
def generate_answer(model, question, sp, max_gen_len=30):
    model.eval()
    # 입력 형식: <s> 질문 </s> 까지 넣어주어 모델이 답변 파트를 시작하게 유도
    tokens = [sp.bos_id()] + sp.EncodeAsIds(preprocess_sentence(question)) + [sp.eos_id()]
    input_ids = torch.tensor([tokens]).to(device)
    
    for _ in range(max_gen_len):
        with torch.no_grad():
            logits = model(input_ids)
            next_token = torch.argmax(logits[:, -1, :], dim=-1)
            
            # 답변이 끝나는 </s>가 나오면 중단
            if next_token.item() == sp.eos_id(): break
            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)
            
    # 생성된 전체 시퀀스에서 질문 파트 이후(답변만) 디코딩
    full_sentence = sp.decode(input_ids[0].tolist())
    # <s> 질문 </s> 이후의 텍스트만 추출
    try:
        answer = full_sentence.split(preprocess_sentence(question))[-1].strip()
    except:
        answer = full_sentence
    return answer

In [13]:
# 테스트 실행
test_sentences = [
    "오늘 날씨 어때?",
    "반가워요",
    "나 배고파",
    "나 졸려",
    "여행 가고 싶다",
    "공부가 너무 어려운데 어떡하지?",
    "요즘 재미있는 영화 추천해줘",
    "이별이란 뭘까?"
]

print("--- [GPT-1 문장 생성 테스트 결과] ---")
model.eval() # 평가 모드 확인

for text in test_sentences:
    # 앞서 정의한 generate 함수 호출
    generated_text = generate(model, text, sp, max_gen_len=20)
    
    print(f"입력: {text}")
    print(f"생성: {generated_text}")
    print("-" * 30)

--- [GPT-1 문장 생성 테스트 결과] ---
입력: 오늘 날씨 어때?
생성: 오늘 날씨 어때? 지났네 .
------------------------------
입력: 반가워요
생성: 반가워요 반갑구만
------------------------------
입력: 나 배고파
생성: 나 배고파 내가 너무 못해
------------------------------
입력: 나 졸려
생성: 나 졸려
------------------------------
입력: 여행 가고 싶다
생성: 여행 가고 싶다
------------------------------
입력: 공부가 너무 어려운데 어떡하지?
생성: 공부가 너무 어려운데 어떡하지? ,
------------------------------
입력: 요즘 재미있는 영화 추천해줘
생성: 요즘 재미있는 영화 추천해줘
------------------------------
입력: 이별이란 뭘까?
생성: 이별이란 뭘까? 엄청 막혀
------------------------------


추가 실험 진행 결과 1

이전에는 Question 데이터만 활용하여 질문을 입력했을때 답에대한 학습을 진행하지 않았기 때문에 질문을 그대로 출력하는 현상을 확인했었다. 이번에는 Question 데이터와 Answer 데이터를 모두 학습을 진행하였고, 이에따라 출력이 개선될 것이라 예상했지만  크게 변경된 것은 없는 것으로 확인되었다.

In [14]:
model_full.train()
print("Q+A 통합 학습을 시작합니다...")
for epoch in range(20): # 대화 패턴 학습을 위해 에폭을 적절히 설정
    epoch_loss = 0
    for x, y in full_dataloader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model_full(x)
        loss = criterion(logits.view(-1, 8000), y.view(-1))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/20 - Loss: {epoch_loss/len(full_dataloader):.4f}")

Q+A 통합 학습을 시작합니다...
Epoch 5/20 - Loss: 1.1563
Epoch 10/20 - Loss: 1.0385
Epoch 15/20 - Loss: 0.9542
Epoch 20/20 - Loss: 0.8974


In [15]:
# 테스트 실행
test_sentences = [
    "오늘 날씨 어때?",
    "반가워요",
    "나 배고파",
    "나 졸려",
    "여행 가고 싶다",
    "공부가 너무 어려운데 어떡하지?",
    "요즘 재미있는 영화 추천해줘",
    "이별이란 뭘까?"
]

print("--- [GPT-1 문장 생성 테스트 결과] ---")
model.eval() # 평가 모드 확인

for text in test_sentences:
    # 앞서 정의한 generate 함수 호출
    generated_text = generate(model, text, sp, max_gen_len=20)
    
    print(f"입력: {text}")
    print(f"생성: {generated_text}")
    print("-" * 30)

--- [GPT-1 문장 생성 테스트 결과] ---
입력: 오늘 날씨 어때?
생성: 오늘 날씨 어때? 지났네 .
------------------------------
입력: 반가워요
생성: 반가워요 반갑구만
------------------------------
입력: 나 배고파
생성: 나 배고파 내가 너무 못해
------------------------------
입력: 나 졸려
생성: 나 졸려
------------------------------
입력: 여행 가고 싶다
생성: 여행 가고 싶다
------------------------------
입력: 공부가 너무 어려운데 어떡하지?
생성: 공부가 너무 어려운데 어떡하지? ,
------------------------------
입력: 요즘 재미있는 영화 추천해줘
생성: 요즘 재미있는 영화 추천해줘
------------------------------
입력: 이별이란 뭘까?
생성: 이별이란 뭘까? 엄청 막혀
------------------------------


추가 실험 진행 2

20에폭을 추가하였지만 그래도 결과는 비슷한 것으로 확인하였다. 아무래도 데이터셋의 문